# Multiperiod HEN Synthesis

**Learning outcome:** Apply multiperiod hen synthesis through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `solver`  
**Expected runtime:** 10 to 90 minutes  
**Optional extras:** hen

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Can one exchanger network serve every operating period without hiding period constraints?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Target periods and synthesize a shared network

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchProblem

base_problem = PinchProblem(
    "Four-stream-Yee-and-Grossmann-1990-1.json"
)
multiperiod_input = base_problem.to_problem_json()
multiperiod_input["options"]["PROBLEM_PERIOD_IDS"] = [
    "base", "turndown"
]
multiperiod_input["options"]["PROBLEM_PERIOD_WEIGHTS"] = [
    0.7, 0.3
]
for stream in multiperiod_input["streams"]:
    duty = stream["heat_flow"]
    stream["heat_flow"] = {
        "values": [duty["value"], 0.8 * duty["value"]],
        "unit": duty["unit"],
    }
    stream["heat_capacity_flowrate"] = None
problem = PinchProblem(
    multiperiod_input, project_name="Four Stream Periods"
)
period_targets = problem.target.all_periods.all_heat_integration()
design = problem.design.multiperiod_heat_exchanger_network(
    stages=[3], best_solutions=1
)
design.top(1)

## Step 2: Inspect the period-specific shared design

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
shared_network = design.network(rank=1)
period_ids = list(problem.period_results)
shared_grid = design.grid(rank=1, period_id=period_ids[0])
shared_network.summary_metrics

## Interpret the result

Inspect the shared network grid in each period and identify the period controlling area and utility demand.

## Adapt this template

Use representative periods and verified weights; increase stages only after a smaller shared-design model is feasible.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.